![image_1780304374570.png](./image_1780304374570.png "image_1780304374570.png")

![image_1780304460253.png](./image_1780304460253.png "image_1780304460253.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("SalesProductDataFrame").getOrCreate()

# Sales dataset
sales_data = [
    (1, 1, 2020, 100, 9.99),
    (2, 1, 2020, 50, 10.99),
    (3, 1, 2021, 200, 8.99),
    (4, 2, 2019, 50, 29.99),
    (5, 2, 2020, 80, 27.99),
    (6, 3, 2021, 30, 49.99),
    (7, 3, 2022, 45, 44.99)
]
sales_columns = ["sale_id", "product_id", "year", "quantity", "price"]
sales_df = spark.createDataFrame(sales_data, sales_columns)

# Product dataset
product_data = [
    (1, "Widget"),
    (2, "Gadget"),
    (3, "Doohickey")
]
product_columns = ["product_id", "product_name"]
product_df = spark.createDataFrame(product_data, product_columns)


In [0]:
result_df=(
    product_df
    .join(sales_df, on='product_id')
    .withColumn("rank",
                f.dense_rank()
                .over(Window.partitionBy(sales_df.product_id)
                      .orderBy(sales_df.year)))
    .filter(f.col("rank") == 1)
    .select(
        product_df.product_name,
        sales_df.year,
        sales_df.quantity,
        sales_df.price
    ).orderBy(product_df.product_name)
)
display(result_df)